In [ ]:
# %%writefile 03_merge_model.ipynb
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Merge Model for Production\n",
    "# Combine LoRA adapter with base model for faster inference"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# INSTALL\n",
    "!pip install -q transformers accelerate peft bitsandbytes"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# IMPORTS\n",
    "import torch\n",
    "import os\n",
    "from transformers import AutoModelForCausalLM, AutoTokenizer\n",
    "from peft import PeftModel\n",
    "\n",
    "# PATHS\n",
    "BASE_MODEL = \"codellama/CodeLlama-7b-Instruct-hf\"\n",
    "ADAPTER_PATH = \"./codellama-coding-chat-final\"\n",
    "MERGED_OUTPUT = \"./production-model-merged\"\n",
    "\n",
    "print(\"🔧 Starting merge process...\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# STEP 1: LOAD BASE MODEL (in higher precision for merging)\n",
    "print(\"Loading base model in FP16...\")\n",
    "\n",
    "tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)\n",
    "\n",
    "base_model = AutoModelForCausalLM.from_pretrained(\n",
    "    BASE_MODEL,\n",
    "    torch_dtype=torch.float16,\n",
    "    device_map=\"auto\",\n",
    "    low_cpu_mem_usage=True,\n",
    ")\n",
    "\n",
    "print(\"✅ Base model loaded\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# STEP 2: LOAD LORA ADAPTER\n",
    "print(\"Loading LoRA adapter...\")\n",
    "\n",
    "model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)\n",
    "\n",
    "print(\"✅ Adapter loaded\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# STEP 3: MERGE AND UNLOAD\n",
    "print(\"Merging LoRA weights into base model...\")\n",
    "print(\"This may take 2-3 minutes...\")\n",
    "\n",
    "merged_model = model.merge_and_unload()\n",
    "\n",
    "print(\"✅ Merge complete!\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# STEP 4: SAVE MERGED MODEL\n",
    "print(f\"Saving merged model to {MERGED_OUTPUT}...\")\n",
    "\n",
    "os.makedirs(MERGED_OUTPUT, exist_ok=True)\n",
    "\n",
    "merged_model.save_pretrained(MERGED_OUTPUT)\n",
    "tokenizer.save_pretrained(MERGED_OUTPUT)\n",
    "\n",
    "print(\"✅ Merged model saved!\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# STEP 5: VERIFY\n",
    "def get_dir_size(path):\n",
    "    total = 0\n",
    "    for dirpath, dirnames, filenames in os.walk(path):\n",
    "        for f in filenames:\n",
    "            fp = os.path.join(dirpath, f)\n",
    "            total += os.path.getsize(fp)\n",
    "    return total / (1024**3)  # GB\n",
    "\n",
    "adapter_size = get_dir_size(ADAPTER_PATH)\n",
    "merged_size = get_dir_size(MERGED_OUTPUT)\n",
    "\n",
    "print(f\"\\n📊 Size Comparison:\")\n",
    "print(f\"  LoRA adapter: {adapter_size:.2f} GB\")\n",
    "print(f\"  Merged model: {merged_size:.2f} GB\")\n",
    "print(f\"\\n✅ Merge successful! Ready for production deployment.\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# STEP 6: QUICK TEST\n",
    "print(\"\\n🧪 Testing merged model...\")\n",
    "\n",
    "test_prompt = \"[INST] Write a Python hello world program [/INST]\"\n",
    "inputs = tokenizer(test_prompt, return_tensors=\"pt\").to(\"cuda\")\n",
    "\n",
    "with torch.no_grad():\n",
    "    outputs = merged_model.generate(\n",
    "        **inputs,\n",
    "        max_new_tokens=100,\n",
    "        temperature=0.7,\n",
    "        do_sample=True,\n",
    "    )\n",
    "\n",
    "response = tokenizer.decode(outputs[0], skip_special_tokens=True)\n",
    "print(f\"Test output:\\n{response}\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}